# Entrega 6 — Componente Avanzado: PCA y t-SNE sobre el Perfil Arancelario

**Curso:** Data Visualization (1ACC0211) — UPC  
**Tema:** Dinámica del comercio mundial: patrones de exportación e importación por país y dimensión arancelaria (1988–2021)

| Código | Nombre |
|---|---|
| U202218912 | Julio Cesar Meza Alfaro |
| U202212675 | Rosa Maria Rodriguez Valencia |
| U202214069 | Braulio Alonso Bartra Sandoval |

**Documento asociado:** [`../docs/entrega6-pca-tsne.md`](../docs/entrega6-pca-tsne.md)

## 0. Configuración y carga de datos

### Razonamiento

El componente avanzado se aplica sobre el **perfil arancelario** (variables AHS/MFN en %), **no** sobre las variables de volumen (Export/Import). Motivo: el `Export_Tier` se definió a partir del volumen de exportación, por lo que un PCA sobre volumen "redescubriría" la variable con la que ya segmentamos (análisis circular). Las variables arancelarias son independientes de cómo se construyó el Tier, así que el análisis es válido.

Esto además cierra un hilo de la Entrega 3: allí se descartó el *pairplot* de 23 variables arancelarias por su colinealidad (descarte D02). PCA es la solución metodológicamente correcta a ese mismo problema.

In [9]:
import pathlib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

ROOT = pathlib.Path("..").resolve()
TABLEAU_DIR = ROOT / "outputs" / "tableau_sources"
PROCESSED_DIR = ROOT / "data" / "processed"

df = pd.read_csv(PROCESSED_DIR / "dataset_limpio_entrega2_consolidado.csv", encoding="utf-8-sig")
dim_country = pd.read_csv(TABLEAU_DIR / "Dim_Country.csv")
tiers = dict(zip(dim_country["Partner Name"], dim_country["Export_Tier"]))

# Variables de perfil arancelario: todas las AHS/MFN expresadas en % (excluye las de importaciones en US$)
tar_cols = [c for c in df.columns if ("AHS" in c or "MFN" in c) and "%" in c and "Imports" not in c]
print(f"Variables de perfil arancelario: {len(tar_cols)}")
for c in tar_cols:
    print("  -", c)

Variables de perfil arancelario: 15
  - AHS Simple Average (%)
  - AHS Weighted Average (%)
  - AHS Dutiable Tariff Lines Share (%)
  - AHS Duty Free Tariff Lines Share (%)
  - AHS Specific Tariff Lines Share (%)
  - AHS AVE Tariff Lines Share (%)
  - AHS MaxRate (%)
  - AHS MinRate (%)
  - MFN Simple Average (%)
  - MFN Weighted Average (%)
  - MFN Dutiable Tariff Lines Share (%)
  - MFN Duty Free Tariff Lines Share (%)
  - MFN Specific Tariff Lines Share (%)
  - MFN AVE Tariff Lines Share (%)
  - MFN MaxRate (%)


## 1. Exclusión de entradas que no son países

### Razonamiento

`Dim_Country` incluye categorías residuales del dataset original (WITS/Comtrade) que **no son economías reales** y distorsionan el análisis: agregados de comercio no atribuido (`Unspecified`, `Special Categories`, `Free Zones`, `Bunkers`, `Neutral Zone`), el agregado `Other Asia, nes` y territorios sin economía (`Antarctica`, `Br. Antr. Terr`, etc.). En particular, `Br. Antr. Terr` genera un outlier degenerado que aplasta el resto de la proyección (distancia ~8× la mediana).

Nota: un filtro por volumen de comercio **no** basta, porque `Bunkers` (combustible de transporte internacional) tiene volumen grande. Se usa una lista explícita.

In [10]:
NO_PAIS = [
    "Unspecified", "Special Categories", "Free Zones", "Other Asia, nes",
    "Bunkers", "Neutral Zone",
    "Antarctica", "Br. Antr. Terr", "Bouvet Island",
    "South Georgia and the South Sandwich Islands", "Norfolk Island", "Pitcairn",
]
print(f"Entradas excluidas del análisis: {len(NO_PAIS)}")

Entradas excluidas del análisis: 12


## 2. PCA + t-SNE + k-means — corte transversal 2021

### Razonamiento

- **Unidad:** un país en 2021 (consistente con el corte del dashboard), descrito por su vector arancelario.
- **Estandarización** (`StandardScaler`, media 0 / desviación 1): imprescindible porque las variables tienen escalas distintas; sin esto, las de mayor rango dominarían los componentes.
- **PCA:** proyección lineal a 2 componentes; se reporta la varianza explicada y las cargas (loadings) para interpretar los ejes.
- **t-SNE** (perplexity=15, `init="pca"`, `random_state=42`): validación no lineal de la estructura local. No se interpretan distancias globales.
- **k-means (k=3):** clustering exploratorio para contrastar contra la segmentación por Tier.

In [11]:
d21 = df[(df["Year"] == 2021) & (~df["Partner Name"].isin(NO_PAIS))].dropna(subset=tar_cols).copy()
tar_2021 = [c for c in tar_cols if d21[c].std() > 0]   # quitar columnas constantes (varianza 0)
print(f"Países con perfil arancelario completo en 2021: {len(d21)} | variables usadas: {len(tar_2021)}")

X = StandardScaler().fit_transform(d21[tar_2021])

pca_full = PCA().fit(X)
var = pca_full.explained_variance_ratio_
print("\nVarianza explicada por componente:", np.round(var[:5], 3))
print(f"PC1+PC2 = {var[:2].sum()*100:.1f}%   |   PC1+PC2+PC3 = {var[:3].sum()*100:.1f}%")

loadings = pd.DataFrame(pca_full.components_[:2].T, index=tar_2021, columns=["PC1", "PC2"])
print("\nTop cargas |PC1| (interpretación: nivel de protección):")
print(loadings["PC1"].abs().sort_values(ascending=False).head(4).round(2))
print("\nTop cargas |PC2| (interpretación: estructura/tipo de arancel):")
print(loadings["PC2"].abs().sort_values(ascending=False).head(4).round(2))

Países con perfil arancelario completo en 2021: 228 | variables usadas: 14

Varianza explicada por componente: [0.237 0.202 0.161 0.1   0.089]
PC1+PC2 = 43.9%   |   PC1+PC2+PC3 = 60.0%

Top cargas |PC1| (interpretación: nivel de protección):
AHS Duty Free Tariff Lines Share (%)    0.51
AHS Dutiable Tariff Lines Share (%)     0.51
AHS Simple Average (%)                  0.44
AHS Specific Tariff Lines Share (%)     0.28
Name: PC1, dtype: float64

Top cargas |PC2| (interpretación: estructura/tipo de arancel):
MFN Specific Tariff Lines Share (%)    0.48
MFN AVE Tariff Lines Share (%)         0.45
AHS MaxRate (%)                        0.44
AHS Weighted Average (%)               0.33
Name: PC2, dtype: float64


In [12]:
# Proyección 2D + t-SNE + clustering
d21[["PC1", "PC2"]] = PCA(n_components=2).fit_transform(X)
d21[["tsne_x", "tsne_y"]] = TSNE(perplexity=15, random_state=42, init="pca").fit_transform(X)
d21["cluster"] = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X)
d21["Export_Tier"] = d21["Partner Name"].map(tiers)

print("Media de PC1 por Tier (corte 2021):")
print(d21.groupby("Export_Tier")["PC1"].mean().round(2))
print("\nClusters (k-means) vs Tier:")
print(pd.crosstab(d21["cluster"], d21["Export_Tier"]))

Media de PC1 por Tier (corte 2021):
Export_Tier
Tier 1 — Grandes Exportadores    -0.41
Tier 2 — Exportadores Medianos   -0.20
Tier 3 — Exportadores Pequeños    0.10
Name: PC1, dtype: float64

Clusters (k-means) vs Tier:
Export_Tier  Tier 1 — Grandes Exportadores  Tier 2 — Exportadores Medianos  \
cluster                                                                      
0                                       26                              23   
1                                        1                               6   
2                                        0                               0   

Export_Tier  Tier 3 — Exportadores Pequeños  
cluster                                      
0                                        33  
1                                        54  
2                                        85  


### Interpretación (corte 2021)

- **PC1 ≈ nivel de protección** (dominan las participaciones de líneas gravadas vs. libres y el promedio AHS).
- **PC2 ≈ estructura del arancel** (específico vs. ad-valorem). Es el componente más difuso: se menciona sin sobre-interpretar.
- En el corte de un solo año, los tiers muestran **solapamiento alto**: la foto anual es ruidosa.
- **Limitación declarada:** PC1+PC2 capturan menos de la mitad de la varianza; la proyección 2D es un resumen aproximado y las conclusiones se plantean como tendencia.

## 3. PCA por año (ejes estables) — fuente para el slider en Tableau

### Razonamiento

Para explorar la evolución con un slider de año en Tableau se necesita que **los ejes signifiquen lo mismo en todos los años**. El procedimiento:

1. **Estandarizar dentro de cada año** → el 0 de cada eje es el país promedio *de ese año* (la posición mide qué tan atípico es un país frente a sus contemporáneos).
2. **Ajustar la rotación (PCA) una sola vez con todos los años juntos** → ejes fijos y comparables entre años. Calcular un PCA distinto por año haría que los componentes cambiaran de significado y los países "saltaran" sin sentido.
3. **Orientar PC1** para que el sentido positivo corresponda a mayor nivel arancelario (correlación positiva con `AHS Simple Average (%)`), de modo que "arriba = perfil más proteccionista/complejo".

**Hallazgo (promediando todos los años):** aparece un gradiente por tamaño — Tier 1 arriba, Tier 3 abajo — que confirma de forma multivariada el hallazgo del dot plot (los grandes exportadores aplican perfiles arancelarios más cargados). t-SNE no aplica aquí: no permite proyectar años nuevos de forma comparable.

In [13]:
d_all = df[~df["Partner Name"].isin(NO_PAIS)].dropna(subset=tar_cols).copy()
tar_all = [c for c in tar_cols if d_all[c].std() > 0]
print(f"Filas país-año: {len(d_all)} | variables: {len(tar_all)}")
cov = d_all.groupby("Year")["Partner Name"].count()
print(f"Cobertura por año: {cov.min()}–{cov.max()} países ({cov.index.min()}–{cov.index.max()})")

# 1) Estandarización POR AÑO: el 0 = promedio de cada año
g = d_all.groupby("Year")
mean_y = g[tar_all].transform("mean")
std_y = g[tar_all].transform("std").replace(0, np.nan)
Z = ((d_all[tar_all] - mean_y) / std_y).fillna(0).values

# 2) Rotación ÚNICA (PCA ajustado con todos los años) -> ejes estables
pca_anio = PCA(n_components=2).fit(Z)
scores = pca_anio.transform(Z)
print("Varianza explicada (PC1, PC2):", np.round(pca_anio.explained_variance_ratio_, 3),
      f"-> {pca_anio.explained_variance_ratio_.sum()*100:.1f}%")

# 3) Orientar PC1: positivo = mayor nivel arancelario
if np.corrcoef(scores[:, 0], d_all["AHS Simple Average (%)"].values)[0, 1] < 0:
    scores[:, 0] *= -1

d_all["PC1"], d_all["PC2"] = scores[:, 0], scores[:, 1]
d_all["Export_Tier"] = d_all["Partner Name"].map(tiers)

Filas país-año: 7463 | variables: 15
Cobertura por año: 188–230 países (1988–2021)
Varianza explicada (PC1, PC2): [0.224 0.167] -> 39.1%


In [14]:
# --- Validación ---
# a) Sanity check con países ancla en 2021: Cuba (arancel ~18%) debe salir arriba; Haiti abajo
chk = d_all[(d_all["Year"] == 2021) & (d_all["Partner Name"].isin(["Cuba", "Haiti", "United States", "Botswana"]))]
print("Check 2021 (PC1 alto = perfil más cargado):")
print(chk[["Partner Name", "PC1", "PC2"]].round(2).to_string(index=False))

# b) Gradiente por Tier promediando todos los años (el hallazgo)
print("\nMedia de PC1 por Tier (todos los años):")
print(d_all.groupby("Export_Tier")["PC1"].mean().round(2))

Check 2021 (PC1 alto = perfil más cargado):
 Partner Name   PC1   PC2
     Botswana -2.80  0.59
         Cuba  4.18  4.04
        Haiti -0.51  3.58
United States  3.33 -1.36

Media de PC1 por Tier (todos los años):
Export_Tier
Tier 1 — Grandes Exportadores     2.30
Tier 2 — Exportadores Medianos    1.17
Tier 3 — Exportadores Pequeños   -0.58
Name: PC1, dtype: float64


## 4. Exportación de fuentes para Tableau

| Archivo | Contenido | Uso en Tableau |
|---|---|---|
| `PCA_tSNE_Paises.csv` | Corte 2021: PC1/PC2, t-SNE, cluster, Tier | Scatter estático del componente avanzado |
| `PCA_Paises_PorAnio.csv` | Todos los años, centrado por año, ejes estables | Scatter con slider/animación por `Year` |

Ambas fuentes son a nivel país (o país-año) y **no se relacionan con `Fact_Trade`** (granularidades distintas → fan-out). Se conectan como fuentes independientes.

In [15]:
out_2021 = d21[["Partner Name", "Export_Tier", "PC1", "PC2", "tsne_x", "tsne_y", "cluster"]]
out_2021.to_csv(TABLEAU_DIR / "PCA_tSNE_Paises.csv", index=False)

out_anio = d_all[["Partner Name", "Year", "PC1", "PC2", "Export_Tier"]].round(4)
out_anio.to_csv(TABLEAU_DIR / "PCA_Paises_PorAnio.csv", index=False)

print(f"PCA_tSNE_Paises.csv     -> {len(out_2021):>5,} filas (corte 2021)")
print(f"PCA_Paises_PorAnio.csv  -> {len(out_anio):>5,} filas (país-año)")

PCA_tSNE_Paises.csv     ->   228 filas (corte 2021)
PCA_Paises_PorAnio.csv  -> 7,463 filas (país-año)


## 5. Conclusiones del componente avanzado

1. **Gradiente por tamaño:** promediando 1988–2021, los grandes exportadores (Tier 1) presentan perfiles arancelarios más proteccionistas/complejos (media PC1 ≈ +2.4) que los pequeños (Tier 3, ≈ −0.7). Confirma de forma **multivariada** el hallazgo del dot plot (Tier 1 = 4.85% vs Tier 3 = 3.41% de arancel promedio en 2021).
2. **Es una tendencia, no una separación:** existe solapamiento entre países individuales y PC1+PC2 capturan una fracción parcial de la varianza total — límite declarado.
3. **Interpretación de ejes:** PC1 = nivel de protección (cuánto cobra + complejidad del sistema); PC2 = estructura del arancel (ad-valorem vs específico), componente difuso que no se sobre-interpreta.
4. **Coherencia del pipeline:** PCA resuelve la colinealidad de las ~15 variables arancelarias que motivó el descarte del pairplot en la Entrega 3, y sus salidas se integran al dashboard como fuentes independientes a nivel país.